In [1]:
import pandas as pd
import xarray as xr
import numpy as np

In [2]:
def load_region_matrix(csv_path, lat_size=360, lon_size=720):
    df = pd.read_csv(csv_path)

    region_matrix = np.empty((lat_size, lon_size), dtype=object)
    region_matrix[:] = None  # 默认填 None

    for _, r in df.iterrows():
        I = int(r["I"]) - 1
        J = int(r["J"]) - 1
        region_matrix[I, J] = r["Rall"]

    return region_matrix


In [ ]:
def split_variable_streaming(da, region_matrix, regions, output_path):
    time = da["time"]
    lat  = da["lat"]
    lon  = da["lon"]

    # 输出 Dataset 初始化（空）
    ds_out = xr.Dataset()

    # 先为每个 region 创建变量（只定义 shape，不写数据）
    for reg in regions:
        var_name = f"{reg}_{da.name}"
        ds_out[var_name] = xr.DataArray(
            np.full((len(time), len(lat), len(lon)), np.nan, dtype=np.float32),
            coords={"time": time, "lat": lat, "lon": lon},
            dims=("time", "lat", "lon")
        )

    # ---- 逐时间片处理 ----
    for ti, t in enumerate(time.values):
        print(f"Processing time = {t}")

        # 只读这个时间片 → 是 360×720，不会爆内存
        da_t = da.sel(time=t).values  # shape = (lat, lon)

        for reg in regions:
            var_name = f"{reg}_{da.name}"

            # mask
            masked = np.where(region_matrix == reg, da_t, np.nan)

            # 写入 Dataset 的对应时间层
            ds_out[var_name][ti, :, :] = masked

    # ---- 最终写入文件 ----
    ds_out.to_netcdf(f"../../NC/{output_path}")
    print(f"[Saved] {output_path}")

In [4]:
def write_region_split_nc(output_path, region_dict):
    ds = xr.Dataset()
    for reg, da in region_dict.items():
        varname = da.name.replace(" ", "_")
        ds[varname] = da
    ds.to_netcdf(output_path)
    print(f"[Saved] {output_path}")

In [ ]:
def run_full_split(csv_path, nc_path, output_prefix):
    # 载入 CSV→region_matrix
    region_matrix = load_region_matrix(csv_path, lat_size=360, lon_size=720)   
    ds = xr.open_dataset(nc_path)

    # 收集地域名称
    regions = set()
    for r in region_matrix.flatten():
        if isinstance(r, str) and r.strip() != "":
            regions.add(r)
    regions = sorted(regions)

    variable_groups = [
        ("region_agri", "region", "agri"),
        ("region_forest", "region", "forest"),
        ("region_grassland", "region", "grassland"),
        ("basin_agri", "basin", "agri"),
        ("basin_forest", "basin", "forest"),
        ("basin_grassland", "basin", "grassland"),
    ]

    for varname, mode, landtype in variable_groups:
        print(f"\n========== Splitting {varname} ==========")

        if varname not in ds:
            print(f"{varname} missing, skip.")
            continue

        da = ds[varname]
        da.name = f"{mode}_{landtype}"

        output_file = f"{mode}_{landtype}_17regions.nc"

        # 使用 streaming 函数
        split_variable_streaming(da, region_matrix, regions, output_file)


In [6]:
CSV_PATH = "../../CSV/gridset/RIJ_17regions.csv"
NC_PATH  = "../../NC/compare.nc"
OUTPUT_PREFIX = "split"

run_full_split(CSV_PATH, NC_PATH, OUTPUT_PREFIX)


========== Splitting region_agri ==========
Processing time = 2005-01-01T00:00:00.000000000
Processing time = 2010-01-01T00:00:00.000000000
Processing time = 2020-01-01T00:00:00.000000000
Processing time = 2030-01-01T00:00:00.000000000
Processing time = 2040-01-01T00:00:00.000000000
Processing time = 2050-01-01T00:00:00.000000000
Processing time = 2060-01-01T00:00:00.000000000
Processing time = 2070-01-01T00:00:00.000000000
Processing time = 2080-01-01T00:00:00.000000000
Processing time = 2090-01-01T00:00:00.000000000
Processing time = 2100-01-01T00:00:00.000000000
[Saved] split_region_agri_17regions.nc

========== Splitting region_forest ==========
Processing time = 2005-01-01T00:00:00.000000000
Processing time = 2010-01-01T00:00:00.000000000
Processing time = 2020-01-01T00:00:00.000000000
Processing time = 2030-01-01T00:00:00.000000000
Processing time = 2040-01-01T00:00:00.000000000
Processing time = 2050-01-01T00:00:00.000000000
Processing time = 2060-01-01T00:00:00.000000000
Proce

KeyboardInterrupt: 